# tBTC TwoCrypto Rebalance Time/Step Study

This notebook forks an RPC via Boa, loads the deployed tBTC TwoCrypto pool from Etherscan, time-travels and performs a small swap to trigger `tweak_price` and observe `price_scale` changes. It then searches for the minimal wait time that causes a rebalancing step (norm/5), and measures how many donation shares are burned at that moment.

Requirements (env):
- `WEB3_PROVIDER_URL` — RPC URL to fork
- `ETHERSCAN_API_KEY` — API key for Etherscan
- Optional: `YB_TBTC_POOL` to override the default tBTC pool address


In [ ]:
import os
import boa

RPC_URL = os.environ.get("WEB3_PROVIDER_URL")
ETHERSCAN_API_KEY = os.environ.get("ETHERSCAN_API_KEY")
assert RPC_URL, "Set WEB3_PROVIDER_URL in environment"
assert ETHERSCAN_API_KEY, "Set ETHERSCAN_API_KEY in environment"

# Fork and set a fresh EOA
try:
    boa.fork(RPC_URL)
except Exception:
    pass
user = boa.env.generate_address()
boa.env.eoa = user
print("Chain:", boa.env.evm.patch.chain_id, "EOA:", user)

In [ ]:
# Load the tBTC TwoCrypto pool via Etherscan
TBTC_POOL = "0xf1F435B05D255a5dBdE37333C0f61DA6F69c6127"
pool = boa.from_etherscan(TBTC_POOL, api_key=ETHERSCAN_API_KEY)
print("Pool:", pool.address)
try:
    print("Name:", pool.name())
except Exception:
    pass
print("Initial price_scale:", pool.price_scale())

In [ ]:
# Discover coins and select the stable side to swap from
c0_addr = pool.coins(0)
c1_addr = pool.coins(1)
c0 = boa.from_etherscan(c0_addr, api_key=ETHERSCAN_API_KEY)
c1 = boa.from_etherscan(c1_addr, api_key=ETHERSCAN_API_KEY)
sym0, sym1 = c0.symbol(), c1.symbol()
dec0, dec1 = int(c0.decimals()), int(c1.decimals())
print("coin0:", c0.address, sym0, "decimals=", dec0)
print("coin1:", c1.address, sym1, "decimals=", dec1)

i_stable, j_other = 0, 1
dec_stable = dec0
print("Stable index:", i_stable, "->", (sym0 if i_stable == 0 else sym1))

In [ ]:
PRECISION = 10**18


def compute_norm(p_oracle: int, p_scale: int) -> int:
    # same as in contract: norm = | (p_oracle*1e18/p_scale) - 1e18 |
    ratio = (p_oracle * PRECISION) // p_scale
    if ratio > PRECISION:
        return ratio - PRECISION
    else:
        return PRECISION - ratio


def compute_step(norm: int, adjustment_step_param: int) -> int:
    # step = max(adjustment_step_param, norm/5)
    return max(adjustment_step_param, norm // 5)


def metrics_snapshot():
    p_scale = int(pool.price_scale())
    # price_oracle() is a view that updates EMA based on current timestamp
    p_oracle = int(pool.price_oracle())
    norm = compute_norm(p_oracle, p_scale)
    adj_param = int(pool.adjustment_step()) if hasattr(pool, "adjustment_step") else 0
    step = compute_step(norm, adj_param)
    allowed = int(pool.allowed_extra_profit()) if hasattr(pool, "allowed_extra_profit") else 0
    return {
        "price_scale": p_scale,
        "price_oracle": p_oracle,
        "norm": norm,
        "adj_param": adj_param,
        "step": step,
        "allowed_extra_profit": allowed,
        "totalSupply": int(pool.totalSupply()),
        "donation_shares": int(pool.donation_shares()),
        "user_supply": int(pool.user_supply())
        if hasattr(pool, "user_supply")
        else int(pool.totalSupply()),
        "virtual_price": int(pool.virtual_price()),
    }


def ensure_balance_and_approve(token, amount: int):
    boa.deal(token, user, amount)
    token.approve(pool, amount, sender=user)


def small_swap_one_usd():
    dx = 10**dec_stable  # 1 unit of the stable coin
    token = c0 if i_stable == 0 else c1
    ensure_balance_and_approve(token, dx)
    dy = pool.exchange(i_stable, 1 - i_stable, dx, 0, sender=user)
    return int(dy)


def swap_into_pool(idx_in, amount: int):
    token = c0 if idx_in == 0 else c1
    ensure_balance_and_approve(token, amount)
    token_out = pool.exchange(idx_in, 1 - idx_in, amount, 0, sender=user)
    return token_out


def donate_to_pool(token_idx: int, amount: int):
    token = c0 if token_idx == 0 else c1
    boa.deal(token, user, amount)
    token.approve(pool, amount, sender=user)
    pool.add_liquidity(
        [amount, 0] if token_idx == 0 else [0, amount],
        0,
        "0x0000000000000000000000000000000000000000",
        True,
        sender=user,
    )


metrics_snapshot()

In [ ]:
# Time-travel one day, execute tiny swap, and observe price_scale
with boa.env.anchor():
    amt = 0.01 * 10**18
    idx = 1
    shares_before = pool.donation_shares()
    donate_to_pool(idx, int(amt))
    shares_after = pool.donation_shares()
    print(
        f"donation shares before: {shares_before/1e18:4.3f}, after: {shares_after/1e18:4.3f}, change: {(shares_after - shares_before)/1e18:4.3f}"
    )
    before = metrics_snapshot()
    boa.env.time_travel(seconds=7 * 86400)
    dy = small_swap_one_usd()
    after = metrics_snapshot()
    changed = after["price_scale"] != before["price_scale"]
    print(f'norm: {before["norm"]}, step: {before["step"]}, (adj_param: {before["adj_param"]})')

    print(f"Donation {amt/1e18:4.5f} of coin{idx} -> Rebalanced? {changed}")
    # print(f'dx=1 unit of stable -> dy={dy}')
    print(
        f'price_scale: {before["price_scale"]/1e18:4.3f} -> {after["price_scale"]/1e18:4.3f}, change: {(after["price_scale"] - before["price_scale"])/1e18:4.3f}'
    )
    print(
        f'virtual price: {before["virtual_price"]/1e18:4.5f} -> {after["virtual_price"]/1e18:4.5f}, change: {(after["virtual_price"] - before["virtual_price"])/1e18:4.5f}'
    )
    print(
        f'shares before: {before["donation_shares"]/1e18:4.3f}, after: {after["donation_shares"]/1e18:4.3f}'
    )
    print(f'shares burned: {(before["donation_shares"] - after["donation_shares"])/1e18:4.3f}')
    print("----\n")

In [ ]:
with boa.env.anchor():
    before = metrics_snapshot()
    # boa.env.time_travel(seconds=2*86400)
    init_amt = 6.8 * 10**18
    amt_in = init_amt
    idx_in = 1
    total_in = 0
    for i in range(10):
        print(f"{i}:")
        boa.env.time_travel(seconds=1)
        # amt_in = 1e5*10**18
        amt_in = int(amt_in)
        total_in += amt_in
        swap1_out = swap_into_pool(idx_in, amt_in)
        print(
            f"swap {amt_in/1e18:4.3f} -> {swap1_out/1e18:4.3f}, avg price: {swap1_out/amt_in:4.3f}"
        )
        boa.env.time_travel(seconds=1)
        swap2_out = swap_into_pool(1 - idx_in, swap1_out)
        print(
            f"swap {swap1_out/1e18:4.3f} -> {swap2_out/1e18:4.3f}, avg price: {swap2_out/swap1_out:4.3f}"
        )
        amt_in = swap2_out
        after = metrics_snapshot()
        changed = after["price_scale"] != before["price_scale"]
        print(f"Rebalanced? {changed}")
        # print(f'dx=1 unit of stable -> dy={dy}')
        print(f'norm: {before["norm"]}, step: {before["step"]}, (adj_param: {before["adj_param"]})')
        print(f'norm_after: {after["norm"]}')
        print(
            f'price_scale: {before["price_scale"]/1e18:4.3f} -> {after["price_scale"]/1e18:4.3f}, change: {(after["price_scale"] - before["price_scale"])/1e18:4.3f}'
        )
        print(
            f'virtual price: {before["virtual_price"]/1e18:4.5f} -> {after["virtual_price"]/1e18:4.5f}, change: {(after["virtual_price"] - before["virtual_price"])/1e18:4.5f}'
        )
        print(
            f'shares before: {before["donation_shares"]/1e18:4.3f}, after: {after["donation_shares"]/1e18:4.3f}'
        )
        print(f'shares burned: {(before["donation_shares"] - after["donation_shares"])/1e18:4.3f}')
        if changed:
            print("Rebalanced!")
            print(
                f"Initial in: {init_amt/1e18:4.3f}, final out: {swap2_out/1e18:4.3f}, loss: {(init_amt - swap2_out)/1e18:4.3f}"
            )
            break

        print("----\n")